# Model Evaluation
Evaluate all three models: crop classifier, phenology mapper, stress detector.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix,
                              jaccard_score, f1_score)
import sys; sys.path.insert(0, '..')
from models.crop_classifier.unet_lstm    import build_model as build_crop
from models.phenology_mapper.transformer import build_model as build_pheno
from models.stress_detector.fusion_cnn   import build_model as build_stress
from inference.infer import CROP_LABELS, STAGE_LABELS, STRESS_LABELS
import yaml

with open('../configs/pipeline_config.yaml') as f:
    cfg = yaml.safe_load(f)['models']


## 1. Phenology Transformer Evaluation

In [ ]:
X_val = np.load('../data/processed/phenology_sequences_val.npy')  # (N, T, 2)
y_val = np.load('../data/processed/phenology_labels_val.npy')     # (N,)

model = build_pheno(cfg['phenology_mapper'])
model.load_state_dict(torch.load('../models/weights/phenology_transformer.pth', map_location='cpu'))
model.eval()

with torch.no_grad():
    logits = model(torch.from_numpy(X_val).float())
    preds  = logits.argmax(dim=1).numpy()

print(classification_report(y_val, preds, target_names=STAGE_LABELS))


In [ ]:
cm = confusion_matrix(y_val, preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=STAGE_LABELS, yticklabels=STAGE_LABELS, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Phenology Confusion Matrix')
plt.tight_layout(); plt.savefig('../data/phenology_cm.png', dpi=150); plt.show()


## 2. Stress Detector Evaluation (sample tile)

In [ ]:
# Load a sample tile pair
import os
opt_files  = sorted(os.listdir('../data/processed/stress_optical/'))
sar_files  = sorted(os.listdir('../data/processed/stress_sar/'))
mask_files = sorted(os.listdir('../data/processed/stress_masks/'))

opt  = np.load(f'../data/processed/stress_optical/{opt_files[0]}').astype(np.float32)
sar  = np.load(f'../data/processed/stress_sar/{sar_files[0]}').astype(np.float32)
mask = np.load(f'../data/processed/stress_masks/{mask_files[0]}').astype(np.int64)

stress_model = build_stress(cfg['stress_detector'])
stress_model.load_state_dict(torch.load('../models/weights/stress_cnn.pth', map_location='cpu'))
stress_model.eval()

with torch.no_grad():
    opt_t = torch.from_numpy(opt).permute(2,0,1).unsqueeze(0).float()
    sar_t = torch.from_numpy(sar).permute(2,0,1).unsqueeze(0).float()
    logits = stress_model(opt_t, sar_t)
    pred_map = logits.argmax(dim=1).squeeze().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, arr, title in zip(axes, [mask, pred_map], ['Ground Truth', 'Prediction']):
    im = ax.imshow(arr, cmap='RdYlGn_r', vmin=0, vmax=2)
    ax.set_title(title); ax.axis('off')
plt.colorbar(im, ax=axes[-1], label='Stress Level')
plt.tight_layout(); plt.savefig('../data/stress_pred_vs_gt.png', dpi=150); plt.show()

print(f'Pixel-wise F1 (macro): {f1_score(mask.ravel(), pred_map.ravel(), average="macro"):.4f}')
